# Patient Education Chatbot

Time estimate: **45 minutes**

## Objectives

After completing this lab, you will be able to:
- Understand how conversational AI can be applied to patient education
- Implement a chatbot using a free, open-source language model
- Create an interactive medical question-answering system
- Evaluate the quality and safety of AI-generated health information
- Apply prompt engineering techniques for medical contexts

## What you will do in this lab

In this lab, you will build a **Patient Education Chatbot** that can answer common health questions, explain medical conditions, and provide general wellness guidance. You'll use a lightweight, open-source language model that runs entirely in Google Colab without requiring expensive hardware or API keys.

## Overview

Patient education is a critical component of healthcare delivery. Studies show that well-informed patients have better health outcomes, higher treatment adherence, and improved satisfaction with care. However, healthcare providers often face time constraints that limit their ability to thoroughly educate patients about their conditions, medications, and lifestyle modifications.

Conversational AI systems offer a promising solution by providing 24/7 access to reliable health information in an accessible, personalized format. These chatbots can answer common questions, explain medical terminology in plain language, and guide patients toward appropriate care resources. When designed carefully with medical expertise and safety guardrails, AI-powered patient education tools can complement human healthcare providers and improve health literacy across diverse populations.

In this lab, you'll explore how to build such a system using modern natural language processing techniques, while also discussing the ethical considerations and limitations of AI in healthcare settings.

## About the dataset

For this lab, you will use a **synthetic medical Q&A dataset** created specifically for educational purposes. The dataset contains:

- **Patient questions**: Common inquiries about symptoms, conditions, medications, and general health
- **Medical categories**: Cardiology, Diabetes, Mental Health, Nutrition, General Wellness
- **Educational responses**: Medically-informed answers written in patient-friendly language

The dataset simulates real patient education scenarios, making it suitable for a learning environment. In a production system, you would use verified medical knowledge bases and content reviewed by licensed healthcare professionals.

## Setup

Let's begin by installing and importing the necessary libraries.

In [ ]:
# Install required packages for your chatbot
# transformers: Hugging Face library for language models
# accelerate: Optimizes model loading and inference
# bitsandbytes: Enables quantization for memory efficiency
!pip install -q transformers accelerate bitsandbytes sentencepiece

In [ ]:
# Import essential Python libraries
import pandas as pd  # For data manipulation
import numpy as np  # For numerical operations
import warnings  # To suppress unnecessary warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

In [ ]:
# Import transformer libraries for the language model
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch  # PyTorch for tensor operations

In [ ]:
# Print library versions for reproducibility
import transformers
print(f"Transformers version: {transformers.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Pandas version: {pd.__version__}")

## Step 1: Create the medical knowledge base

First, you'll create a synthetic dataset containing common patient questions and educational responses across different medical categories.

In [ ]:
# Create synthetic medical Q&A dataset
medical_qa_data = {
    'category': [
        'Cardiology', 'Cardiology', 'Diabetes', 'Diabetes', 'Mental Health',
        'Mental Health', 'Nutrition', 'Nutrition', 'General Wellness', 'General Wellness',
        'Cardiology', 'Diabetes', 'Mental Health', 'Nutrition', 'General Wellness'
    ],
    'question': [
        'What is high blood pressure and why is it dangerous?',
        'How can I lower my cholesterol naturally?',
        'What are the early signs of diabetes?',
        'How does insulin work in the body?',
        'What is the difference between anxiety and stress?',
        'How can I improve my sleep quality?',
        'What foods should I eat to boost my immune system?',
        'Is intermittent fasting safe for everyone?',
        'How much water should I drink daily?',
        'What are the benefits of regular exercise?',
        'What lifestyle changes help prevent heart disease?',
        'Can diabetes be reversed with diet?',
        'When should I seek help for depression?',
        'What vitamins are most important for adults?',
        'How many hours of sleep do adults need?'
    ],
    'answer': [
        'High blood pressure (hypertension) occurs when blood pushes too hard against artery walls. It\'s dangerous because it can damage blood vessels, leading to heart disease, stroke, and kidney problems. Many people have no symptoms, making regular checkups important.',
        'Natural ways to lower cholesterol include eating more fiber (oats, beans, fruits), choosing healthy fats (olive oil, avocados), exercising regularly, maintaining a healthy weight, and avoiding trans fats. Omega-3 fatty acids from fish can also help.',
        'Early diabetes signs include increased thirst, frequent urination, unexplained weight loss, fatigue, blurred vision, and slow-healing wounds. However, type 2 diabetes often develops gradually with mild symptoms. Regular screening is recommended for at-risk individuals.',
        'Insulin is a hormone that helps glucose (sugar) enter your cells for energy. After eating, your pancreas releases insulin to manage rising blood sugar. In diabetes, either the body doesn\'t make enough insulin (type 1) or cells become resistant to it (type 2).',
        'Stress is a response to external pressures or demands, usually temporary. Anxiety is a persistent feeling of worry or fear that may occur without an obvious trigger. While stress typically resolves when the situation ends, anxiety can be ongoing and may require professional treatment.',
        'To improve sleep: maintain a consistent schedule, create a dark and cool bedroom, avoid screens before bed, limit caffeine and alcohol, exercise regularly (but not close to bedtime), and establish a relaxing bedtime routine like reading or meditation.',
        'Immune-boosting foods include citrus fruits (vitamin C), red bell peppers, broccoli, garlic, ginger, spinach, yogurt with probiotics, almonds, turmeric, and green tea. A balanced diet with varied colorful fruits and vegetables provides essential nutrients for immune function.',
        'Intermittent fasting isn\'t safe for everyone. Avoid it if you\'re pregnant, breastfeeding, under 18, have diabetes, a history of eating disorders, or certain medical conditions. Always consult your healthcare provider before starting any fasting regimen.',
        'Most adults need about 8 glasses (64 ounces) of water daily, but needs vary based on activity level, climate, and health status. A good indicator is pale yellow urine. You need more water when exercising, in hot weather, or if you\'re ill.',
        'Regular exercise strengthens your heart, improves circulation, helps control weight, reduces disease risk, enhances mood, boosts energy, promotes better sleep, and increases longevity. Aim for 150 minutes of moderate activity or 75 minutes of vigorous activity weekly.',
        'Heart-healthy lifestyle changes include: not smoking, eating a Mediterranean-style diet rich in fruits and vegetables, exercising regularly, maintaining healthy weight, managing stress, limiting alcohol, controlling blood pressure and cholesterol, and getting adequate sleep.',
        'Type 2 diabetes can sometimes be put into remission through significant lifestyle changes including weight loss, healthy eating, and regular exercise. However, it requires ongoing management. Type 1 diabetes cannot be reversed as it involves autoimmune destruction of insulin-producing cells.',
        'Seek help for depression if you experience persistent sadness, loss of interest in activities, changes in sleep or appetite, difficulty concentrating, feelings of worthlessness, or thoughts of self-harm for more than two weeks. Depression is treatable—early intervention improves outcomes.',
        'Important vitamins for adults include: Vitamin D (bone health, immune function), B vitamins (energy, brain function), Vitamin C (immune support, antioxidant), Vitamin A (vision, immune health), and Vitamin E (antioxidant). A balanced diet usually provides adequate vitamins.',
        'Most adults need 7-9 hours of sleep per night. Individual needs vary, but consistently getting less than 7 hours is associated with increased risk of obesity, diabetes, heart disease, and mental health issues. Quality matters as much as quantity.'
    ]
}

In [ ]:
# Convert dictionary to pandas DataFrame for easy manipulation
medical_df = pd.DataFrame(medical_qa_data)

# Display the shape of our dataset
print(f"Dataset contains {len(medical_df)} Q&A pairs")
print(f"\nCategories: {medical_df['category'].unique()}")

In [ ]:
# Display sample questions from each category
print("Sample Questions by Category:\n")
for category in medical_df['category'].unique():
    # Filter questions by category
    sample = medical_df[medical_df['category'] == category]['question'].iloc[0]
    print(f"[{category}] {sample}")

## Step 2: Load the language model

You'll use **TinyLlama-1.1B**, a small but capable open-source language model that:
- Runs efficiently in Google Colab's free tier
- Requires less than 3GB of RAM
- Is completely free and open-source
- Can generate coherent, contextual responses

In [ ]:
# Check if GPU is available (makes inference faster)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Define the model name from Hugging Face
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print("Loading model... This may take 1-2 minutes.")
print("Model: TinyLlama 1.1B - optimized for efficient inference")

In [ ]:
# Load the tokenizer (converts text to numbers the model understands)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set padding token to avoid warnings
tokenizer.pad_token = tokenizer.eos_token

print("✓ Tokenizer loaded successfully")

In [ ]:
# Load the language model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,  # Use half precision on GPU
    device_map="auto",  # Automatically place model on available device
    low_cpu_mem_usage=True  # Optimize for low memory usage
)

print("✓ Model loaded successfully")

## Step 3: Create the medical chatbot system

Now you'll create a system prompt and chatbot function that guides the model to provide safe, educational medical information.

In [ ]:
# Define the system prompt that sets the chatbot's behavior
SYSTEM_PROMPT = """You are a helpful patient education assistant. Your role is to:
1. Provide clear, accurate health information in simple language
2. Encourage patients to consult healthcare providers for diagnosis and treatment
3. Never provide specific medical diagnoses or prescribe treatments
4. Be empathetic and supportive
5. Cite when professional medical attention is needed

Always remind users that this information is educational and not a substitute for professional medical advice."""

print("System prompt configured for safe medical education")

In [ ]:
def create_medical_prompt(question, context=""):
    """
    Creates a formatted prompt for the medical chatbot.

    Args:
        question (str): The patient's question
        context (str): Optional context from knowledge base

    Returns:
        str: Formatted prompt for the model
    """
    # Format the prompt with system instructions and user question
    prompt = f"""<|system|>
{SYSTEM_PROMPT}</s>
<|user|>
{question}</s>
<|assistant|>
"""

    return prompt

In [ ]:
def generate_response(question, max_length=300, temperature=0.7):
    """
    Generates a response to a patient's question.

    Args:
        question (str): Patient's health question
        max_length (int): Maximum length of response
        temperature (float): Creativity level (0.0-1.0)

    Returns:
        str: Generated response
    """
    # Create the formatted prompt
    prompt = create_medical_prompt(question)

    # Tokenize the input text
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate response using the model
    with torch.no_grad():  # Disable gradient calculation for inference
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,  # Maximum tokens to generate
            temperature=temperature,  # Controls randomness
            do_sample=True,  # Enable sampling for varied responses
            top_p=0.9,  # Nucleus sampling parameter
            pad_token_id=tokenizer.eos_token_id  # Padding token
        )

    # Decode the generated tokens back to text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the assistant's response
    response = response.split("<|assistant|>")[-1].strip()

    return response

## Step 4: Test the chatbot

Let's test the chatbot with various medical questions to see how it performs.

In [ ]:
# Test question 1: General health inquiry
test_question_1 = "What is high blood pressure?"

print(f"Question: {test_question_1}\n")
print("Generating response...\n")

# Generate and display response
response_1 = generate_response(test_question_1)
print(f"Chatbot: {response_1}")

In [ ]:
# Test question 2: Symptom inquiry
test_question_2 = "I have frequent headaches. What could cause this?"

print(f"Question: {test_question_2}\n")
print("Generating response...\n")

# Generate and display response
response_2 = generate_response(test_question_2)
print(f"Chatbot: {response_2}")

In [ ]:
# Test question 3: Lifestyle advice
test_question_3 = "How can I improve my sleep quality?"

print(f"Question: {test_question_3}\n")
print("Generating response...\n")

# Generate and display response
response_3 = generate_response(test_question_3)
print(f"Chatbot: {response_3}")

## Step 5: Create an interactive chat interface

Now let's create a simple interactive interface where you can ask multiple questions in a conversation.

In [ ]:
def interactive_chatbot():
    """
    Creates an interactive chat session with the medical chatbot.
    Type 'quit' or 'exit' to end the conversation.
    """
    print("="*60)
    print("Patient Education Chatbot")
    print("="*60)
    print("Ask me health questions! Type 'quit' or 'exit' to end.\n")
    print("Remember: This is educational only, not medical advice.\n")

    # Continue conversation until user quits
    while True:
        # Get user input
        user_question = input("You: ").strip()

        # Check if user wants to exit
        if user_question.lower() in ['quit', 'exit', 'bye']:
            print("\nChatbot: Take care! Remember to consult your healthcare provider for medical concerns.")
            break

        # Skip empty inputs
        if not user_question:
            continue

        # Generate and display response
        print("\nChatbot: ", end="")
        response = generate_response(user_question)
        print(response)
        print()  # Add blank line for readability

In [ ]:
# Run the interactive chatbot
# Uncomment the line below to start chatting
# interactive_chatbot()

## Step 6: Evaluate response quality

Let's evaluate how well your chatbot performs on your knowledge base questions.

In [ ]:
# Select a few questions from your dataset to test
test_questions = medical_df['question'].head(3).tolist()

# Store results for comparison
results = []

print("Testing chatbot on sample questions...\n")
print("="*60)

In [ ]:
# Generate responses for each test question
for i, question in enumerate(test_questions, 1):
    print(f"\n[Test {i}/3]")
    print(f"Q: {question}")

    # Generate response
    response = generate_response(question, max_length=200)

    print(f"A: {response}")
    print("-"*60)

    # Store for later analysis
    results.append({
        'question': question,
        'response': response,
        'length': len(response.split())
    })

In [ ]:
# Display response statistics
results_df = pd.DataFrame(results)

print("\nResponse Statistics:")
print(f"Average response length: {results_df['length'].mean():.1f} words")
print(f"Shortest response: {results_df['length'].min()} words")
print(f"Longest response: {results_df['length'].max()} words")

## Step 7: Implement safety checks

Medical chatbots need safety mechanisms to handle emergency situations and inappropriate queries.

In [ ]:
def check_emergency_keywords(question):
    """
    Checks if the question contains emergency-related keywords.

    Args:
        question (str): User's question

    Returns:
        bool: True if emergency keywords detected
    """
    # Define emergency keywords
    emergency_keywords = [
        'chest pain', 'heart attack', 'stroke', 'can\'t breathe',
        'difficulty breathing', 'severe bleeding', 'unconscious',
        'suicide', 'overdose', 'poisoning', 'severe pain'
    ]

    # Check if any keyword appears in the question
    question_lower = question.lower()
    for keyword in emergency_keywords:
        if keyword in question_lower:
            return True

    return False

In [ ]:
def safe_chatbot_response(question):
    """
    Generates a safe response with emergency handling.

    Args:
        question (str): User's question

    Returns:
        str: Safe response with emergency warnings if needed
    """
    # Check for emergency situations
    if check_emergency_keywords(question):
        return """ EMERGENCY ALERT

If you are experiencing a medical emergency, please:
1. Call 911 (or your local emergency number) immediately
2. Do not wait for online advice
3. If someone is with you, ask them to call while you wait

For mental health emergencies:
- National Suicide Prevention Lifeline: 988 (US)
- Crisis Text Line: Text HOME to 741741

This chatbot cannot handle emergencies. Please seek immediate help."""

    # Generate normal response for non-emergency questions
    response = generate_response(question)

    # Add disclaimer
    disclaimer = "\n\nThis information is for educational purposes only. Please consult a healthcare provider for medical advice."

    return response + disclaimer

In [ ]:
# Test the safety check
emergency_question = "I'm having severe chest pain"

print(f"Testing emergency detection...\n")
print(f"Question: {emergency_question}\n")
print(safe_chatbot_response(emergency_question))

In [ ]:
# Test with a normal question
normal_question = "What foods are good for heart health?"

print(f"\nTesting normal question...\n")
print(f"Question: {normal_question}\n")
print(safe_chatbot_response(normal_question))

## Exercises

Now it's your turn to extend the chatbot's capabilities!

### Exercise 1: Add more medical categories

Expand the knowledge base by adding 3 new Q&A pairs in a new medical category (e.g., "Pediatrics", "Women's Health", "Respiratory Health").

In [ ]:
# Your code here
# Add new rows to the medical_df DataFrame

# Hint: Use pd.concat() or df.loc[] to add new rows


<details>
<summary>Hint</summary>

Create a new dictionary with the same structure as `medical_qa_data`, then use `pd.concat([medical_df, new_df], ignore_index=True)` to combine them.

</details>

<details>
<summary>Solution</summary>

```python
# Create new medical Q&A pairs for Respiratory Health
new_qa_data = {
    'category': ['Respiratory Health', 'Respiratory Health', 'Respiratory Health'],
    'question': [
        'What is asthma and how is it managed?',
        'How can I prevent the common cold?',
        'What are the symptoms of pneumonia?'
    ],
    'answer': [
        'Asthma is a chronic condition where airways become inflamed and narrow, causing difficulty breathing. Management includes avoiding triggers, using prescribed inhalers (controller and rescue medications), monitoring symptoms, and following an asthma action plan with your doctor.',
        'Prevent colds by washing hands frequently, avoiding close contact with sick people, not touching your face, getting adequate sleep, managing stress, eating a nutritious diet, and staying hydrated. No supplement or medication can completely prevent colds.',
        'Pneumonia symptoms include cough (may produce phlegm), fever, chills, shortness of breath, chest pain when breathing or coughing, fatigue, and confusion (especially in older adults). Severe symptoms require immediate medical attention.'
    ]
}

# Create new DataFrame
new_df = pd.DataFrame(new_qa_data)

# Combine with existing data
medical_df = pd.concat([medical_df, new_df], ignore_index=True)

# Verify the addition
print(f"Updated dataset size: {len(medical_df)} Q&A pairs")
print(f"Categories: {medical_df['category'].unique()}")
```

</details>

### Exercise 2: Implement response filtering

Create a function that checks if the chatbot's response inappropriately claims to diagnose or prescribe treatment. If detected, replace with a safer response.

In [ ]:
# Your code here


<details>
<summary>Hint</summary>

Create a list of inappropriate phrases like "you have", "I diagnose", "you should take". Use a loop or any() function to check if these appear in the response. Return a generic safe response if detected.

</details>

<details>
<summary>Solution</summary>

```python
def filter_inappropriate_response(response):
    """
    Checks if response contains inappropriate medical claims.
    
    Args:
        response (str): Generated chatbot response
    
    Returns:
        str: Filtered safe response
    """
    # Define inappropriate phrases that suggest diagnosis or prescription
    inappropriate_phrases = [
        'you have',
        'you are diagnosed',
        'i diagnose',
        'you should take',
        'take this medication',
        'you need to take',
        'my diagnosis is'
    ]
    
    # Convert response to lowercase for checking
    response_lower = response.lower()
    
    # Check if any inappropriate phrase is present
    for phrase in inappropriate_phrases:
        if phrase in response_lower:
            # Return safe alternative
            return """I cannot provide specific diagnoses or prescribe treatments.
Based on your question, I recommend:

1. Schedule an appointment with your healthcare provider
2. Describe your symptoms in detail to them
3. Follow their professional medical advice

Healthcare providers can properly examine you, order necessary tests,
and provide appropriate treatment based on your individual situation."""
    
    # Return original response if safe
    return response

# Test the filter
test_response = "Based on your symptoms, you have bronchitis and should take antibiotics."
print("Original:", test_response)
print("\nFiltered:", filter_inappropriate_response(test_response))
```

</details>

### Exercise 3: Create a conversation history feature

Implement a conversation history that stores previous questions and answers, allowing the chatbot to reference earlier parts of the conversation.

In [ ]:
# Your code here


<details>
<summary>Hint</summary>

Store each Q&A pair as a dictionary in the `conversation_history` list. When generating a new response, include the last 2-3 exchanges in the prompt to provide context.

</details>

<details>
<summary>Solution</summary>

```python
# Initialize conversation history
conversation_history = []

def chatbot_with_memory(question, max_history=3):
    """
    Generates response using conversation history for context.
    
    Args:
        question (str): User's question
        max_history (int): Maximum number of previous exchanges to include
    
    Returns:
        str: Generated response
    """
    # Build context from recent conversation history
    context = ""
    recent_history = conversation_history[-max_history:] if len(conversation_history) > 0 else []
    
    for exchange in recent_history:
        context += f"User: {exchange['question']}\nAssistant: {exchange['response']}\n\n"
    
    # Create prompt with conversation context
    prompt = f"""<|system|>
{SYSTEM_PROMPT}

Previous conversation:
{context}</s>
<|user|>
{question}</s>
<|assistant|>
"""
    
    # Tokenize and generate response
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("<|assistant|>")[-1].strip()
    
    # Add to conversation history
    conversation_history.append({
        'question': question,
        'response': response
    })
    
    return response

# Test the memory feature
print("Testing conversation memory...\n")
print("Q1: What is diabetes?")
response1 = chatbot_with_memory("What is diabetes?")
print(f"A1: {response1}\n")

print("Q2: How can I prevent it?")  # "it" refers to diabetes from previous question
response2 = chatbot_with_memory("How can I prevent it?")
print(f"A2: {response2}")
```

</details>

### Exercise 4: Implement multilingual support

Research and implement a simple translation layer that allows the chatbot to respond in different languages. You can use the `googletrans` library or another translation API.

In [ ]:
# Your code here
# Install translation library: !pip install googletrans==4.0.0-rc1
# Create function to translate responses to user's preferred language


<details>
<summary>Hint</summary>

Install the googletrans library, import Translator, generate the response in English first, then use `translator.translate(text, dest='language_code')` to translate it.

</details>

<details>
<summary>Solution</summary>

```python
# Install translation library
!pip install -q googletrans==4.0.0-rc1

from googletrans import Translator

# Initialize translator
translator = Translator()

def multilingual_chatbot(question, target_language='en'):
    """
    Generates response and translates to target language.
    
    Args:
        question (str): User's question (in any language)
        target_language (str): ISO language code (e.g., 'es', 'fr', 'zh-cn')
    
    Returns:
        str: Translated response
    """
    # Translate question to English if needed
    detected = translator.detect(question)
    if detected.lang != 'en':
        question_en = translator.translate(question, dest='en').text
    else:
        question_en = question
    
    # Generate response in English
    response_en = generate_response(question_en, max_length=150)
    
    # Translate response to target language
    if target_language != 'en':
        response = translator.translate(response_en, dest=target_language).text
    else:
        response = response_en
    
    return response

# Test multilingual feature
print("Testing Spanish translation...\n")
question_es = "¿Qué es la presión arterial alta?"
print(f"Pregunta: {question_es}")
response_es = multilingual_chatbot(question_es, target_language='es')
print(f"Respuesta: {response_es}")
```

</details>

## Congratulations!

You have successfully completed this lab on **Patient Education Chatbot development**. You learned how to apply natural language processing and prompt engineering to design safe, efficient conversational AI systems for healthcare. This lab emphasized ethical safeguards, resource-efficient deployment, and transparent communication, helping you build chatbots that support patient education while maintaining trust and safety in medical contexts.

## Authors

Ramesh Sannareddy

Copyright © 2025 SkillUp. All rights reserved.